# 🌊 Challenge A: Resilient Release Scheduling — Falcon Reservoir
## OQI Quantum Hackathon · Junio 2026 — V2 (Corregido)

---

**Mejoras V2:**
- ✅ Fix ventana de datos: Large ahora usa 52 semanas reales
- ✅ Baselines corregidos: respetan restricción anti-hoarding
- ✅ QUBO mejorado: C_crit con encoding cuadrático completo
- ✅ SA clásico mejorado: multi-start + más iteraciones
- ✅ QAOA con Qiskit: solver cuántico de compuertas
- ✅ Local search post-processing después del QUBO

In [ ]:
# ═══════════════════════════════════════════════════
# 0. INSTALACIÓN DE DEPENDENCIAS
# ═══════════════════════════════════════════════════
!pip install -q dimod dwave-neal
!pip install -q qiskit qiskit-optimization qiskit-algorithms qiskit-aer

In [ ]:
import pandas as pd
import numpy as np
import time, os, glob, warnings
import matplotlib.pyplot as plt
import dimod, neal

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9', 'xtick.color': '#8b949e',
    'ytick.color': '#8b949e', 'grid.color': '#21262d',
    'grid.alpha': 0.6, 'font.size': 11, 'axes.titlesize': 14,
    'legend.facecolor': '#161b22', 'legend.edgecolor': '#30363d',
    'legend.labelcolor': '#c9d1d9',
})

COL = {'hist':'#8b949e','rule':'#f0883e','adapt':'#a371f7',
       'sa':'#58a6ff','qubo':'#3fb950','qaoa':'#f778ba',
       'smin':'#f85149','smax':'#1f6feb','accent':'#d2a8ff'}

print('✅ Dependencias cargadas')

## 1. Carga de Datos IBWC

In [ ]:
# Subir archivos en Colab
try:
    from google.colab import files
    print('📁 Sube los 3 CSVs principales del hackathon:')
    print('   1. Total Storage (Web-Daily) @08461200')
    print('   2. Discharge Total.Change-in-Storage @08461200')
    print('   3. Discharge.Best Available @08461300')
    uploaded = files.upload()
    print(f'\n✅ {len(uploaded)} archivos subidos')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('ℹ️ No estás en Colab — usando rutas locales')

In [ ]:
def find_file(pattern):
    for f in glob.glob('./*') + glob.glob('/content/*'):
        if pattern.lower() in os.path.basename(f).lower():
            return f
    raise FileNotFoundError(f'No encontrado: {pattern}')

try:
    STORAGE_FILE   = find_file('total storage.web-daily')
    DELTA_S_FILE   = find_file('change-in-storage')
    DISCHARGE_FILE = find_file('discharge.best available')
    print('✅ Archivos detectados:')
    for f in [STORAGE_FILE, DELTA_S_FILE, DISCHARGE_FILE]:
        print(f'   {os.path.basename(f)}')
except FileNotFoundError as e:
    print(f'❌ {e}\n👉 Ajusta las rutas manualmente abajo')

# === RUTAS MANUALES (descomenta si falla auto-detección) ===
# STORAGE_FILE   = 'DataSetExport-Total Storage.Web-Daily-ac-ft@08461200-Instantaneous-TCM-20260622185130.csv'
# DELTA_S_FILE   = 'DataSetExport-Discharge Total.Change-in-Storage@08461200-Instantaneous-TCM-20260622185956.csv'
# DISCHARGE_FILE = 'DataSetExport-Discharge.Best Available@08461300-Instantaneous-m^3 s-20260622190542.csv'

In [ ]:
def load_csv(path, val_col='value'):
    df = pd.read_csv(path, skiprows=1)
    df.columns = ['timestamp', val_col, 'grade', 'approval', 'interp']
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df[val_col] = pd.to_numeric(df[val_col], errors='coerce')
    return df.dropna(subset=['timestamp', val_col]).sort_values('timestamp').reset_index(drop=True)

print('📂 Cargando...')
storage_df = load_csv(STORAGE_FILE, 'storage_tcm')
delta_s_df = load_csv(DELTA_S_FILE, 'delta_S_tcm')
release_df = load_csv(DISCHARGE_FILE, 'discharge_m3s')

for name, df in [('Storage', storage_df), ('ΔStorage', delta_s_df), ('Discharge', release_df)]:
    print(f'   {name}: {len(df):,} reg  ({df.timestamp.min().date()} → {df.timestamp.max().date()})')
print('✅ Datos cargados')

## 2. Preprocesamiento — CORREGIDO

**Fix V2:** Se busca la ventana crítica SOLO dentro del período donde los 3 datasets se solapan (2012+), y se garantizan 52 semanas para Large.

In [ ]:
# Resampleo semanal
release_weekly = (
    release_df.set_index('timestamp')['discharge_m3s']
    .resample('W').mean() * 7 * 86400 / 1000  # m³/s → TCM/semana
).reset_index().rename(columns={'discharge_m3s': 'R_obs_tcm'}).dropna()

delta_s_weekly = (
    delta_s_df.set_index('timestamp')['delta_S_tcm']
    .resample('W').sum()
).reset_index().dropna()

storage_weekly = (
    storage_df.set_index('timestamp')['storage_tcm']
    .resample('W').mean()
).reset_index().rename(columns={'storage_tcm': 'S_obs_tcm'}).dropna()

# Merge: solo semanas donde TODOS los datasets tienen datos
weekly = (
    release_weekly
    .merge(delta_s_weekly, on='timestamp', how='inner')
    .merge(storage_weekly, on='timestamp', how='inner')
    .sort_values('timestamp').reset_index(drop=True)
)
print(f'Semanas combinadas disponibles: {len(weekly)}')
print(f'Rango: {weekly.timestamp.min().date()} → {weekly.timestamp.max().date()}')

# FIX V2: Buscar ventana crítica de 52 semanas DENTRO de los datos combinados
T_MAX_NEEDED = 52
if len(weekly) < T_MAX_NEEDED:
    T_MAX_NEEDED = len(weekly)
    print(f'⚠ Solo {len(weekly)} semanas disponibles, ajustando T_max={T_MAX_NEEDED}')

# Rolling: buscar las 52 semanas con menor storage promedio
s_vals = weekly['S_obs_tcm'].values
best_start, min_avg = 0, float('inf')
for i in range(len(s_vals) - T_MAX_NEEDED + 1):
    avg = np.mean(s_vals[i:i + T_MAX_NEEDED])
    if avg < min_avg:
        min_avg, best_start = avg, i

df_window = weekly.iloc[best_start:best_start + T_MAX_NEEDED].reset_index(drop=True)
print(f'\n🎯 Ventana más crítica ({T_MAX_NEEDED} sem):')
print(f'   {df_window.timestamp.iloc[0].date()} → {df_window.timestamp.iloc[-1].date()}')
print(f'   Storage promedio: {min_avg:,.0f} TCM')

R_obs_full  = df_window['R_obs_tcm'].values
dS_obs_full = df_window['delta_S_tcm'].values
S_obs_full  = df_window['S_obs_tcm'].values
T_available = len(R_obs_full)
print(f'   T disponible: {T_available} semanas ✅')

## 3. Parámetros Oficiales

In [ ]:
S_MAX = storage_df['storage_tcm'].max()
S_MIN = 0.25 * S_MAX
eta   = 0.10

# S_init: storage real al inicio de la ventana seleccionada
S_INIT = S_obs_full[0]

R_median = np.median(R_obs_full)
delta_u  = 0.25 * R_median
u_max    = 2 * delta_u

levels_5 = np.array([-2, -1, 0, 1, 2]) * delta_u
levels_3 = np.array([-2, 0, 2]) * delta_u

def get_weights(T, u_max, S_min):
    w1 = 1.0 / ((T+1) * S_min**2)
    w2 = 0.1 / (T * u_max**2)
    w3 = 0.1 / ((T-1) * (2*u_max)**2)
    return w1, w2, w3

INSTANCES = {
    'Small':  {'T': 12, 'L': 3},
    'Medium': {'T': 26, 'L': 5},  # ← BENCHMARK OFICIAL
    'Large':  {'T': 52, 'L': 5},
}

print(f'S_max  = {S_MAX:>14,.0f} TCM')
print(f'S_min  = {S_MIN:>14,.0f} TCM (25%)')
print(f'S_init = {S_INIT:>14,.0f} TCM ({S_INIT/S_MAX*100:.1f}%)')
print(f'Δu     = {delta_u:>14,.0f} TCM/sem')
print(f'u_max  = {u_max:>14,.0f} TCM/sem')
print(f'η      = {eta}')
print(f'Niveles (L=5): {np.round(levels_5, 1)}')

## 4. Funciones Core: SRS + Baselines Corregidos

In [ ]:
def compute_SRS(u_seq, S0, dS_obs, R_obs, S_min, S_max, u_max, w1, w2, w3, eta=0.10):
    """Calcula SRS completo con verificación de factibilidad."""
    T = len(u_seq)
    S = S0
    C_crit = C_dev = C_smooth = 0.0
    feasible = True
    storages = [S0]
    violations = []

    for t in range(T):
        u = u_seq[t]
        R = R_obs[t] + u
        S_new = S + dS_obs[t] - u

        if R < -1e-6:
            feasible = False; violations.append(f't={t}: R<0')
        if abs(u) > u_max + 1e-6:
            feasible = False; violations.append(f't={t}: |u|>u_max')
        if S_new < -1e-6:
            feasible = False; violations.append(f't={t}: S<0')
        if S_new > S_max + 1e-6:
            feasible = False; violations.append(f't={t}: S>S_max')

        C_crit  += max(0, S_min - S_new)**2
        C_dev   += u**2
        if t > 0:
            C_smooth += (u - u_seq[t-1])**2
        S = S_new
        storages.append(S)

    total_R = sum(R_obs[:T])
    if total_R > 0 and abs(sum(u_seq)) > eta * total_R:
        feasible = False; violations.append('Anti-hoarding')

    SRS = -(w1*C_crit + w2*C_dev + w3*C_smooth)
    comp = {'C_crit':C_crit, 'C_dev':C_dev, 'C_smooth':C_smooth,
            'w1_Cc':w1*C_crit, 'w2_Cd':w2*C_dev, 'w3_Cs':w3*C_smooth,
            'violations':violations}
    return SRS, storages, feasible, comp

print('✅ compute_SRS definido')

In [ ]:
# ═══════════════════════════════════════════════════
# BASELINES CORREGIDOS (V2): respetan anti-hoarding
# ═══════════════════════════════════════════════════

def threshold_rule_v2(S0, dS_obs, R_obs, S_min, delta_u, T, eta=0.10):
    """Threshold rule CON anti-hoarding compliance."""
    u_seq, S = [], S0
    cum_u = 0.0
    total_R = sum(R_obs[:T])
    limit = eta * total_R if total_R > 0 else float('inf')

    for t in range(T):
        if S < S_min and abs(cum_u - delta_u) <= limit:
            adj = -delta_u
        else:
            adj = 0.0
        # Asegurar R >= 0
        if R_obs[t] + adj < 0:
            adj = 0.0
        u_seq.append(adj)
        cum_u += adj
        S = S + dS_obs[t] - adj
    return np.array(u_seq)


def adaptive_rule_v2(S0, dS_obs, R_obs, S_min, S_max, delta_u, T, eta=0.10):
    """Adaptive rule CON anti-hoarding compliance."""
    u_seq, S = [], S0
    cum_u = 0.0
    total_R = sum(R_obs[:T])
    limit = eta * total_R if total_R > 0 else float('inf')

    for t in range(T):
        if S < 0.15 * S_max:
            adj = -2 * delta_u
        elif S < S_min:
            adj = -1 * delta_u
        elif S > 0.75 * S_max:
            adj = +1 * delta_u
        else:
            adj = 0.0

        # Anti-hoarding check
        if abs(cum_u + adj) > limit:
            adj = 0.0
        # R >= 0
        if R_obs[t] + adj < 0:
            adj = max(-R_obs[t], -2*delta_u)
            if R_obs[t] + adj < 0:
                adj = 0.0

        u_seq.append(adj)
        cum_u += adj
        S = S + dS_obs[t] - adj
    return np.array(u_seq)

print('✅ Baselines V2 definidos (con anti-hoarding)')

## 5. Simulated Annealing Clásico — Mejorado

**Mejoras V2:** Multi-start (5 semillas), 200K iteraciones, movimientos de 2 semanas simultáneas.

In [ ]:
def classical_SA(dS_obs, R_obs, S0, S_min, S_max, u_max,
                 w1, w2, w3, levels, T,
                 n_iter=200000, T_init=1.0, T_final=0.0005,
                 seed=42, eta=0.10, n_starts=5):
    """SA clásico mejorado con multi-start."""
    L = len(levels)
    global_best_srs = -float('inf')
    global_best_u = np.zeros(T)
    all_history = []

    for start in range(n_starts):
        rng = np.random.RandomState(seed + start * 7)

        # Inicio variado: 50% desde u=0, 50% aleatorio
        if start == 0:
            current = np.full(T, L // 2, dtype=int)
        else:
            current = rng.randint(0, L, size=T)

        current_u = levels[current]
        current_srs, _, current_feas, _ = compute_SRS(
            current_u, S0, dS_obs, R_obs, S_min, S_max, u_max, w1, w2, w3, eta)
        if not current_feas:
            current_srs -= 1e6

        best = current.copy()
        best_srs = current_srs
        best_u = current_u.copy()
        history = [best_srs]

        for i in range(n_iter):
            temp = T_init * (T_final / T_init) ** (i / n_iter)
            neighbor = current.copy()

            # 70% cambiar 1 semana, 30% cambiar 2 semanas
            n_changes = 1 if rng.random() < 0.7 else 2
            for _ in range(n_changes):
                t_c = rng.randint(T)
                neighbor[t_c] = rng.randint(L)

            neighbor_u = levels[neighbor]
            n_srs, _, n_feas, _ = compute_SRS(
                neighbor_u, S0, dS_obs, R_obs, S_min, S_max, u_max, w1, w2, w3, eta)
            if not n_feas:
                n_srs -= 1e6

            delta = n_srs - current_srs
            if delta > 0 or rng.random() < np.exp(delta / max(temp, 1e-15)):
                current = neighbor
                current_u = neighbor_u
                current_srs = n_srs
                if current_srs > best_srs:
                    best = current.copy()
                    best_srs = current_srs
                    best_u = current_u.copy()

            if i % 2000 == 0:
                history.append(best_srs)

        if best_srs > global_best_srs:
            global_best_srs = best_srs
            global_best_u = best_u.copy()
        all_history.append(history)

    return global_best_u, global_best_srs, all_history[0]

print('✅ SA clásico mejorado (multi-start) definido')

## 6. Formulación QUBO — Mejorada V2

**Mejora clave:** C_crit ahora usa encoding cuadrático completo en lugar de linealización.

Para cada semana $t$ donde el storage baseline $B(t) < S_{min}$:

$$D(t) = S_{min} - B(t) + \sum_{\tau < t} u(\tau)$$

$$C_{crit}(t) = D(t)^2 = d(t)^2 + 2 d(t) \sum a_{\tau,k} x_{\tau,k} + \left(\sum a_{\tau,k} x_{\tau,k}\right)^2$$

Esto genera tanto términos lineales (diagonal) como cuadráticos (off-diagonal) en la matriz Q.

In [ ]:
def build_qubo_v2(dS_obs, R_obs, S0, S_min, S_max, u_max,
                  w1, w2, w3, levels, T, lambda_oh=None, eta=0.10):
    """
    QUBO V2: one-hot encoding con C_crit cuadrático completo.
    Variables: x[t,k] ∈ {0,1},  N = T × L
    """
    L = len(levels)
    N = T * L
    Q = {}
    idx = lambda t, k: t * L + k

    def add_Q(i, j, val):
        if abs(val) < 1e-20: return
        key = (min(i,j), max(i,j))
        Q[key] = Q.get(key, 0) + val

    # Auto-escala lambda_oh
    if lambda_oh is None:
        obj_scale = max(abs(w2 * u_max**2), abs(w3 * (2*u_max)**2), 1e-10)
        lambda_oh = 1e3 * obj_scale  # V2: reducido de 1e4 a 1e3

    # ─── 1. One-Hot: (Σ_k x[t,k] - 1)² ───
    for t in range(T):
        for k in range(L):
            i = idx(t, k)
            add_Q(i, i, lambda_oh * (-1))
            for k2 in range(k+1, L):
                add_Q(i, idx(t, k2), lambda_oh * 2)

    # ─── 2. C_dev: w2 * Σ_t u(t)² ───
    for t in range(T):
        for k in range(L):
            add_Q(idx(t,k), idx(t,k), w2 * levels[k]**2)

    # ─── 3. C_smooth: w3 * Σ_t (u(t)-u(t-1))² ───
    for t in range(1, T):
        for k1 in range(L):
            for k2 in range(L):
                i, j = idx(t, k1), idx(t-1, k2)
                add_Q(i, j, w3 * (levels[k1] - levels[k2])**2)

    # ─── 4. C_crit CUADRÁTICO (V2 FIX) ───
    # Baseline storage sin ajustes
    B = np.zeros(T + 1)
    B[0] = S0
    for t in range(T):
        B[t+1] = B[t] + dS_obs[t]

    for t_eval in range(1, T + 1):
        d_t = S_min - B[t_eval]  # deficit baseline
        if d_t <= -T * u_max:  # imposible que ajustes lo lleven a zona crítica
            continue

        # S_opt(t_eval) = B(t_eval) - Σ_{τ<t_eval} u(τ)
        # deficit D = S_min - S_opt = d_t + Σ_{τ<t_eval} u(τ)
        # u(τ) = Σ_k levels[k] * x[τ,k]
        #
        # Penalizamos D² = [d_t + Σ_{τ,k} levels[k]*x[τ,k]]²
        # Solo si d_t > 0 (baseline ya en déficit)

        if d_t > 0:
            # Término lineal: 2*d_t*levels[k] (para cada τ < t_eval)
            for tau in range(min(t_eval, T)):
                for k in range(L):
                    i = idx(tau, k)
                    add_Q(i, i, w1 * 2 * d_t * levels[k])

            # Término cuadrático: levels[k1]*levels[k2] (cruzados)
            for tau1 in range(min(t_eval, T)):
                for k1 in range(L):
                    i = idx(tau1, k1)
                    # Diagonal (mismo variable)
                    add_Q(i, i, w1 * levels[k1]**2)
                    # Off-diagonal (diferentes variables)
                    for tau2 in range(tau1 + 1, min(t_eval, T)):
                        for k2 in range(L):
                            j = idx(tau2, k2)
                            add_Q(i, j, w1 * 2 * levels[k1] * levels[k2])
                    # Mismo tau, diferente k
                    for k2 in range(k1 + 1, L):
                        j = idx(tau1, k2)
                        add_Q(i, j, w1 * 2 * levels[k1] * levels[k2])

    # ─── 5. Anti-hoarding suave ───
    lambda_ah = w2 * 0.05
    for t1 in range(T):
        for k1 in range(L):
            i = idx(t1, k1)
            for t2 in range(t1, T):
                for k2 in range(L):
                    j = idx(t2, k2)
                    add_Q(i, j, lambda_ah * levels[k1] * levels[k2])

    return Q, N

print('✅ QUBO V2 builder definido (con C_crit cuadrático)')

In [ ]:
def decode_qubo(sample, T, L, levels):
    """Decodifica solución one-hot → u(t)."""
    u = []
    for t in range(T):
        chosen = [k for k in range(L) if sample.get(t*L+k, 0) == 1]
        if len(chosen) == 1:
            u.append(levels[chosen[0]])
        elif len(chosen) == 0:
            u.append(0.0)
        else:
            u.append(levels[min(chosen, key=lambda k: abs(levels[k]))])
    return np.array(u)


def local_search(u_init, dS_obs, R_obs, S0, S_min, S_max,
                 u_max, w1, w2, w3, levels, eta=0.10, max_iter=5000):
    """Post-QUBO: local search greedy para mejorar solución."""
    T = len(u_init)
    L = len(levels)
    best_u = u_init.copy()
    best_srs, _, best_feas, _ = compute_SRS(
        best_u, S0, dS_obs, R_obs, S_min, S_max, u_max, w1, w2, w3, eta)
    if not best_feas:
        best_srs -= 1e6

    improved = True
    iters = 0
    while improved and iters < max_iter:
        improved = False
        iters += 1
        for t in range(T):
            for lev in levels:
                if abs(lev - best_u[t]) < 1e-10:
                    continue
                trial = best_u.copy()
                trial[t] = lev
                trial_srs, _, trial_feas, _ = compute_SRS(
                    trial, S0, dS_obs, R_obs, S_min, S_max, u_max, w1, w2, w3, eta)
                if not trial_feas:
                    trial_srs -= 1e6
                if trial_srs > best_srs + 1e-12:
                    best_u = trial
                    best_srs = trial_srs
                    improved = True

    return best_u, best_srs


def repair_feasibility(u_seq, R_obs, S0, dS_obs, S_min, S_max, u_max, eta, levels):
    """Repara infactibilidad con greedy."""
    T = len(u_seq)
    u = u_seq.copy()

    for t in range(T):
        u[t] = np.clip(u[t], -u_max, u_max)
        u[t] = levels[np.argmin(np.abs(levels - u[t]))]

    S = S0
    for t in range(T):
        if R_obs[t] + u[t] < 0:
            valid = levels[levels >= -R_obs[t] - 1e-6]
            u[t] = valid[0] if len(valid) > 0 else 0.0
        S_new = S + dS_obs[t] - u[t]
        if S_new < 0:
            cands = levels[levels < u[t] + 1e-6]
            if len(cands) > 0: u[t] = cands[0]
        S = S + dS_obs[t] - u[t]

    total_R = sum(R_obs[:T])
    total_u = sum(u)
    if total_R > 0 and abs(total_u) > eta * total_R:
        excess = abs(total_u) - eta * total_R
        sign = np.sign(total_u)
        for i in np.argsort(-np.abs(u)):
            if excess <= 0: break
            old = u[i]
            if sign > 0 and old > 0:
                u[i] = levels[np.argmin(np.abs(levels - max(0, old-excess)))]
                excess -= (old - u[i])
            elif sign < 0 and old < 0:
                u[i] = levels[np.argmin(np.abs(levels - min(0, old+excess)))]
                excess -= (u[i] - old)
    return u

print('✅ Decode, local search, y repair definidos')

## 7. QAOA con Qiskit

Usamos QAOA (Quantum Approximate Optimization Algorithm) de IBM Qiskit.
- **Tiny** (T=4, L=3, N=12 qubits): factible en simulador
- **Small** (T=12, L=3, N=36 qubits): intento con menos shots
- **Medium/Large**: demasiado grande para simulador → solo SA

In [ ]:
def solve_qaoa(Q_dict, T, L, levels, dS_obs, R_obs, S0, S_min, S_max,
               u_max, w1, w2, w3, eta=0.10, reps=1, maxiter=100):
    """
    Resuelve QUBO con QAOA de Qiskit.
    Retorna: u_seq, SRS, runtime, info_dict
    """
    N = T * L

    try:
        from qiskit_optimization import QuadraticProgram
        from qiskit_optimization.algorithms import MinimumEigenOptimizer
        from qiskit_algorithms import QAOA
        from qiskit_algorithms.optimizers import COBYLA
        from qiskit.primitives import Sampler
    except ImportError:
        try:
            from qiskit_optimization import QuadraticProgram
            from qiskit_optimization.algorithms import MinimumEigenOptimizer
            from qiskit_algorithms.minimum_eigensolvers import QAOA
            from qiskit_algorithms.optimizers import COBYLA
            from qiskit.primitives import Sampler
        except ImportError as e:
            print(f'   ⚠ Qiskit no disponible: {e}')
            return None, None, 0, {'error': str(e)}

    # Construir QuadraticProgram
    qp = QuadraticProgram('FalconQUBO')
    for i in range(N):
        qp.binary_var(f'x{i}')

    linear = {}
    quadratic = {}
    for (i, j), v in Q_dict.items():
        if i == j:
            linear[f'x{i}'] = linear.get(f'x{i}', 0) + v
        else:
            quadratic[(f'x{i}', f'x{j}')] = quadratic.get((f'x{i}', f'x{j}'), 0) + v

    qp.minimize(linear=linear, quadratic=quadratic)

    # QAOA
    print(f'     QAOA: {N} qubits, reps={reps}, maxiter={maxiter}')
    sampler = Sampler()
    optimizer = COBYLA(maxiter=maxiter)

    try:
        qaoa = QAOA(sampler=sampler, optimizer=optimizer, reps=reps)
    except TypeError:
        # Fallback for older API
        qaoa = QAOA(optimizer=optimizer, reps=reps, sampler=sampler)

    eigen_optimizer = MinimumEigenOptimizer(qaoa)

    t0 = time.time()
    result = eigen_optimizer.solve(qp)
    runtime = time.time() - t0

    # Decodificar
    sample = {i: int(result.x[i]) for i in range(N)}
    u_raw = decode_qubo(sample, T, L, levels)
    u_rep = repair_feasibility(u_raw, R_obs, S0, dS_obs, S_min, S_max, u_max, eta, levels)
    u_final, srs_final = local_search(
        u_rep, dS_obs, R_obs, S0, S_min, S_max, u_max, w1, w2, w3, levels, eta)

    srs, storages, feas, comp = compute_SRS(
        u_final, S0, dS_obs, R_obs, S_min, S_max, u_max, w1, w2, w3, eta)

    info = {'fval': result.fval, 'status': str(result.status),
            'raw_u': u_raw, 'repaired_u': u_rep,
            'n_vars': N, 'reps': reps}

    return u_final, srs, runtime, info

print('✅ QAOA solver definido')

## 8. Ejecución Completa — Todas las Instancias

In [ ]:
results = {}

for inst_name, cfg in INSTANCES.items():
    T = min(cfg['T'], T_available)
    L = cfg['L']
    lvls = levels_3 if L == 3 else levels_5

    R_T  = R_obs_full[:T]
    dS_T = dS_obs_full[:T]
    w1, w2, w3 = get_weights(T, u_max, S_MIN)
    args = (S_INIT, dS_T, R_T, S_MIN, S_MAX, u_max, w1, w2, w3)

    print(f'\n{"="*65}')
    print(f'  {inst_name} (T={T}, L={L}, N={T*L} vars, espacio={L}^{T}={L**T:.1e})')
    print(f'{"="*65}')

    # ── Baselines ──
    u_hist = np.zeros(T)
    SRS_hist, S_hist, f_hist, c_hist = compute_SRS(u_hist, *args)

    u_rule = threshold_rule_v2(S_INIT, dS_T, R_T, S_MIN, delta_u, T, eta)
    SRS_rule, S_rule, f_rule, c_rule = compute_SRS(u_rule, *args)

    u_adapt = adaptive_rule_v2(S_INIT, dS_T, R_T, S_MIN, S_MAX, delta_u, T, eta)
    SRS_adapt, S_adapt, f_adapt, c_adapt = compute_SRS(u_adapt, *args)

    # ── SA Clásico ──
    print(f'  🔥 SA clásico (5 starts × 200K iter)...')
    t0 = time.time()
    u_sa, _, sa_hist = classical_SA(
        dS_T, R_T, S_INIT, S_MIN, S_MAX, u_max, w1, w2, w3,
        lvls, T, n_iter=200000, n_starts=5, seed=42)
    time_sa = time.time() - t0
    SRS_sa, S_sa, f_sa, c_sa = compute_SRS(u_sa, *args)

    # ── QUBO + dimod SA ──
    print(f'  ⚛️  QUBO V2 ({T*L} vars)...')
    t0 = time.time()
    Q_dict, N_v = build_qubo_v2(
        dS_T, R_T, S_INIT, S_MIN, S_MAX, u_max, w1, w2, w3, lvls, T)
    time_build = time.time() - t0
    print(f'     Construido: {len(Q_dict):,} entradas ({time_build:.2f}s)')

    bqm = dimod.BinaryQuadraticModel.from_qubo(Q_dict)
    sampler = neal.SimulatedAnnealingSampler()
    t0 = time.time()
    response = sampler.sample(bqm, num_reads=500, num_sweeps=10000, seed=42)
    time_solve = time.time() - t0

    u_qubo_raw = decode_qubo(response.first.sample, T, L, lvls)
    u_qubo_rep = repair_feasibility(u_qubo_raw, R_T, S_INIT, dS_T, S_MIN, S_MAX, u_max, eta, lvls)
    print(f'  🔍 Local search post-QUBO...')
    u_qubo, _ = local_search(u_qubo_rep, dS_T, R_T, S_INIT, S_MIN, S_MAX, u_max, w1, w2, w3, lvls, eta)
    SRS_qubo, S_qubo, f_qubo, c_qubo = compute_SRS(u_qubo, *args)

    # ── Almacenar resultados ──
    res = {
        'T': T, 'L': L,
        'SRS_hist': SRS_hist, 'S_hist': S_hist, 'comp_hist': c_hist, 'u_hist': u_hist,
        'SRS_rule': SRS_rule, 'S_rule': S_rule, 'feas_rule': f_rule, 'comp_rule': c_rule, 'u_rule': u_rule,
        'SRS_adapt': SRS_adapt, 'S_adapt': S_adapt, 'feas_adapt': f_adapt, 'comp_adapt': c_adapt, 'u_adapt': u_adapt,
        'SRS_sa': SRS_sa, 'S_sa': S_sa, 'feas_sa': f_sa, 'comp_sa': c_sa, 'u_sa': u_sa,
        'SRS_qubo': SRS_qubo, 'S_qubo': S_qubo, 'feas_qubo': f_qubo, 'comp_qubo': c_qubo, 'u_qubo': u_qubo,
        'sa_hist': sa_hist, 'R_T': R_T, 'dS_T': dS_T,
        'w1': w1, 'w2': w2, 'w3': w3,
        'time_sa': time_sa, 'time_build': time_build, 'time_solve': time_solve,
    }

    # ── QAOA (solo para Small o menor) ──
    if T * L <= 36:
        print(f'  🔮 QAOA Qiskit ({T*L} qubits)...')
        reps_val = 2 if T*L <= 18 else 1
        maxiter_val = 150 if T*L <= 18 else 80
        u_qaoa, srs_qaoa, time_qaoa, qaoa_info = solve_qaoa(
            Q_dict, T, L, lvls, dS_T, R_T, S_INIT, S_MIN, S_MAX,
            u_max, w1, w2, w3, eta, reps=reps_val, maxiter=maxiter_val)
        if u_qaoa is not None:
            SRS_qaoa, S_qaoa, f_qaoa, c_qaoa = compute_SRS(u_qaoa, *args)
            res.update({
                'SRS_qaoa': SRS_qaoa, 'S_qaoa': S_qaoa,
                'feas_qaoa': f_qaoa, 'comp_qaoa': c_qaoa,
                'u_qaoa': u_qaoa, 'time_qaoa': time_qaoa,
                'qaoa_info': qaoa_info,
            })

    results[inst_name] = res

    # ── Imprimir tabla ──
    print(f'\n  {"Método":<26} {"SRS":>12} {"ΔSRS":>12} {"Fact.":>6} {"Time":>8}')
    print(f'  {"-"*66}')
    rows = [
        ('Historical Replay',  SRS_hist,  f_hist,  0),
        ('Threshold Rule V2',  SRS_rule,  f_rule,  0),
        ('Adaptive Rule V2',   SRS_adapt, f_adapt, 0),
        ('Classical SA (5×200K)', SRS_sa, f_sa,    time_sa),
        ('QUBO+SA (dimod V2)', SRS_qubo,  f_qubo,  time_build+time_solve),
    ]
    if 'SRS_qaoa' in res:
        rows.append(('QAOA (Qiskit)',  res['SRS_qaoa'], res['feas_qaoa'], res['time_qaoa']))
    for name, srs, feas, t_e in rows:
        d = srs - SRS_hist
        print(f'  {name:<26} {srs:>12.6f} {d:>+12.6f} {"✅" if feas else "❌":>6} {t_e:>7.1f}s')

print('\n\n✅ Todas las instancias completadas')

## 9. Gráfica — Almacenamiento Histórico Falcon

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(storage_df['timestamp'], storage_df['storage_tcm'], color=COL['sa'], lw=0.8)
ax.axhline(S_MIN, color=COL['smin'], lw=2, ls=':', label=f'S_min={S_MIN:,.0f}')
ax.axhline(S_MAX, color=COL['smax'], lw=1, ls=':', alpha=0.4, label=f'S_max={S_MAX:,.0f}')
ax.fill_between(storage_df['timestamp'], 0, S_MIN, alpha=0.06, color=COL['smin'])
ax.set(xlabel='Fecha', ylabel='Almacenamiento (TCM)',
       title='Embalse Falcon — Almacenamiento Histórico (1953–presente)')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 10. Gráfica — Trayectorias de Almacenamiento

In [ ]:
def plot_traj(inst):
    r = results[inst]
    T = r['T']
    w = np.arange(T+1)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10),
                                    gridspec_kw={'height_ratios': [3,1]})

    # Storage trajectories
    ax1.plot(w, r['S_hist'], color=COL['hist'], lw=2, ls='--', label=f'Historical ({r["SRS_hist"]:.4f})')
    ax1.plot(w, r['S_rule'], color=COL['rule'], lw=2, label=f'Threshold ({r["SRS_rule"]:.4f}) {"✅" if r["feas_rule"] else "❌"}')
    ax1.plot(w, r['S_adapt'], color=COL['adapt'], lw=2, label=f'Adaptive ({r["SRS_adapt"]:.4f}) {"✅" if r["feas_adapt"] else "❌"}')
    ax1.plot(w, r['S_sa'], color=COL['sa'], lw=2.5, label=f'Classical SA ({r["SRS_sa"]:.4f})')
    ax1.plot(w, r['S_qubo'], color=COL['qubo'], lw=3, marker='o', ms=3, label=f'QUBO+SA ({r["SRS_qubo"]:.4f})')
    if 'S_qaoa' in r:
        ax1.plot(w, r['S_qaoa'], color=COL['qaoa'], lw=2.5, marker='s', ms=3, label=f'QAOA ({r["SRS_qaoa"]:.4f})')

    ax1.axhline(S_MIN, color=COL['smin'], lw=2, ls=':', alpha=0.8)
    ax1.fill_between(w, 0, S_MIN, alpha=0.06, color=COL['smin'])
    ax1.text(1, S_MIN*0.4, 'ZONA CRÍTICA', color=COL['smin'], fontsize=10, fontweight='bold', alpha=0.5)
    ax1.set(ylabel='Almacenamiento (TCM)', xlim=(0, T),
            title=f'Trayectorias — {inst} (T={T})')
    ax1.legend(loc='upper right', fontsize=8); ax1.grid(True, alpha=0.3)

    # Adjustments
    wk = np.arange(T)
    bw = 0.18
    ax2.bar(wk-1.5*bw, r['u_rule'], bw, color=COL['rule'], alpha=0.7, label='Threshold')
    ax2.bar(wk-0.5*bw, r['u_adapt'], bw, color=COL['adapt'], alpha=0.7, label='Adaptive')
    ax2.bar(wk+0.5*bw, r['u_sa'], bw, color=COL['sa'], alpha=0.7, label='SA')
    ax2.bar(wk+1.5*bw, r['u_qubo'], bw, color=COL['qubo'], alpha=0.7, label='QUBO')
    ax2.axhline(0, color='#555', lw=0.5)
    ax2.axhline(u_max, color=COL['smin'], lw=1, ls=':', alpha=0.4)
    ax2.axhline(-u_max, color=COL['smin'], lw=1, ls=':', alpha=0.4)
    ax2.set(ylabel='u(t) TCM', xlabel='Semana')
    ax2.legend(ncol=4, fontsize=8); ax2.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

for inst in results:
    plot_traj(inst)

## 11. Gráfica — Comparación SRS

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

insts = list(results.keys())
meths = ['Historical','Threshold','Adaptive','SA','QUBO+SA','QAOA']
keys  = ['SRS_hist','SRS_rule','SRS_adapt','SRS_sa','SRS_qubo','SRS_qaoa']
cols  = [COL['hist'],COL['rule'],COL['adapt'],COL['sa'],COL['qubo'],COL['qaoa']]
x = np.arange(len(insts))
w = 0.13

for i, (m, k, c) in enumerate(zip(meths, keys, cols)):
    vals = [results[inst].get(k, None) for inst in insts]
    vals = [v if v is not None else 0 for v in vals]
    has_val = [results[inst].get(k) is not None for inst in insts]
    if any(has_val):
        ax1.bar(x + i*w - 2.5*w, vals, w, label=m, color=c, alpha=0.85)
        deltas = [results[inst].get(k, results[inst]['SRS_hist']) - results[inst]['SRS_hist'] for inst in insts]
        ax2.bar(x + (i-1)*w - 2*w, deltas, w, label=m, color=c, alpha=0.85)

ax1.set(xticks=x, ylabel='SRS', title='SRS por Método e Instancia')
ax1.set_xticklabels(insts); ax1.legend(fontsize=8); ax1.grid(True, axis='y', alpha=0.3)

ax2.set(xticks=x, ylabel='ΔSRS', title='Mejora sobre Historical')
ax2.set_xticklabels(insts); ax2.axhline(0, color='#555', lw=0.5)
ax2.legend(fontsize=8); ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

## 12. Gráfica — Descomposición del Costo (Medium)

In [ ]:
def plot_costs(inst):
    r = results[inst]
    meths = ['Historical','Threshold','Adaptive','SA','QUBO+SA']
    comps = [r['comp_hist'], r['comp_rule'], r['comp_adapt'], r['comp_sa'], r['comp_qubo']]
    if 'comp_qaoa' in r:
        meths.append('QAOA')
        comps.append(r['comp_qaoa'])

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(meths))
    w = 0.25
    ax.bar(x-w, [c['w1_Cc'] for c in comps], w, color='#f85149', alpha=0.85, label='w₁·C_crit')
    ax.bar(x,   [c['w2_Cd'] for c in comps], w, color='#58a6ff', alpha=0.85, label='w₂·C_dev')
    ax.bar(x+w, [c['w3_Cs'] for c in comps], w, color='#d2a8ff', alpha=0.85, label='w₃·C_smooth')
    ax.set(xticks=x, ylabel='Costo ponderado (menor=mejor)',
           title=f'Descomposición del Costo — {inst}')
    ax.set_xticklabels(meths); ax.legend(); ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

plot_costs('Medium')

## 13. Gráfica — Convergencia SA y Perfil de Liberaciones

In [ ]:
# Convergencia SA
r = results['Medium']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

h = r['sa_hist']
ax1.plot(np.arange(len(h))*2000, h, color=COL['sa'], lw=2, label='Classical SA')
ax1.axhline(r['SRS_hist'], color=COL['hist'], ls='--', label=f'Historical: {r["SRS_hist"]:.4f}')
ax1.axhline(r['SRS_qubo'], color=COL['qubo'], ls='-.', lw=2, label=f'QUBO+SA: {r["SRS_qubo"]:.4f}')
if 'SRS_qaoa' in r:
    ax1.axhline(r['SRS_qaoa'], color=COL['qaoa'], ls='-.', lw=2, label=f'QAOA: {r["SRS_qaoa"]:.4f}')
ax1.set(xlabel='Iteraciones', ylabel='Mejor SRS', title='Convergencia SA — Medium')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# Perfil liberaciones
T = r['T']
wk = np.arange(T)
ax2.fill_between(wk, 0, r['R_T'], alpha=0.1, color=COL['hist'])
ax2.plot(wk, r['R_T'], color=COL['hist'], lw=2, ls='--', label='R_obs')
ax2.plot(wk, r['R_T']+r['u_sa'], color=COL['sa'], lw=2, label='R_opt SA')
ax2.plot(wk, r['R_T']+r['u_qubo'], color=COL['qubo'], lw=2.5, marker='o', ms=3, label='R_opt QUBO')
ax2.axhline(0, color=COL['smin'], lw=1, alpha=0.3)
ax2.set(xlabel='Semana', ylabel='Liberación (TCM/sem)', title='Perfil de Liberaciones — Medium')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 14. Gráfica — Estructura QUBO y Escalado

In [ ]:
# Matriz QUBO (Small para visualizar)
r_s = results['Small']
T_s, L_s = r_s['T'], r_s['L']
lvls_s = levels_3 if L_s == 3 else levels_5
Q_vis, N_vis = build_qubo_v2(
    r_s['dS_T'], r_s['R_T'], S_INIT, S_MIN, S_MAX, u_max,
    r_s['w1'], r_s['w2'], r_s['w3'], lvls_s, T_s)

Q_dense = np.zeros((N_vis, N_vis))
for (i,j), v in Q_vis.items():
    Q_dense[i,j] = v
    if i != j: Q_dense[j,i] = v

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# QUBO matrix
Q_log = np.sign(Q_dense) * np.log1p(np.abs(Q_dense))
ax1.imshow(Q_log, cmap='RdBu_r', aspect='equal')
ax1.set_title(f'QUBO Small ({N_vis}×{N_vis})', fontweight='bold')
for t in range(T_s+1): ax1.axhline(t*L_s-0.5, color='w', lw=0.3, alpha=0.3); ax1.axvline(t*L_s-0.5, color='w', lw=0.3, alpha=0.3)

# Scaling
T_vals = [results[i]['T'] for i in results]
L_vals = [results[i]['L'] for i in results]
search = [l**t for t,l in zip(T_vals, L_vals)]
ax2.semilogy(T_vals, search, 'D-', color=COL['accent'], lw=2.5, ms=10)
for t,s,inst in zip(T_vals, search, results.keys()):
    ax2.annotate(f'{inst}\n{s:.1e}', (t,s), textcoords='offset points',
                 xytext=(0,15), ha='center', fontsize=9, color=COL['accent'])
ax2.set(xlabel='T (semanas)', ylabel='L^T', title='Escalado Exponencial')
ax2.grid(True, alpha=0.3)

# Runtime
insts = list(results.keys())
sa_t = [results[i]['time_sa'] for i in insts]
qubo_t = [results[i]['time_build']+results[i]['time_solve'] for i in insts]
x = np.arange(len(insts))
ax3.bar(x-0.2, sa_t, 0.35, color=COL['sa'], alpha=0.85, label='SA Clásico')
ax3.bar(x+0.2, qubo_t, 0.35, color=COL['qubo'], alpha=0.85, label='QUBO+SA')
if any('time_qaoa' in results[i] for i in insts):
    qaoa_t = [results[i].get('time_qaoa', 0) for i in insts]
    ax3.bar(x+0.55, qaoa_t, 0.3, color=COL['qaoa'], alpha=0.85, label='QAOA')
ax3.set(xticks=x, ylabel='Tiempo (s)', title='Runtime'); ax3.set_xticklabels(insts)
ax3.legend(fontsize=9); ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

## 15. Reporte Final

In [ ]:
print('╔' + '═'*74 + '╗')
print('║' + '  REPORTE FINAL V2 — Challenge A: Falcon Reservoir'.center(74) + '║')
print('║' + '  OQI Quantum Hackathon · Junio 2026'.center(74) + '║')
print('╠' + '═'*74 + '╣')

for inst, r in results.items():
    n_qubits = r['T'] * r['L']
    print(f'║  {inst} (T={r["T"]}, L={r["L"]}, N={n_qubits} qubits)'.ljust(74) + '║')
    print('║  ' + '-'*70 + '  ║')
    print(f'║  {"Método":<24} {"SRS":>11} {"ΔSRS":>11} {"Fact":>5} {"Time":>7}  ║')
    print('║  ' + '-'*70 + '  ║')

    rows = [
        ('Historical',     r['SRS_hist'],  True,            0),
        ('Threshold V2',   r['SRS_rule'],  r['feas_rule'],  0),
        ('Adaptive V2',    r['SRS_adapt'], r['feas_adapt'], 0),
        ('Classical SA',   r['SRS_sa'],    r['feas_sa'],    r['time_sa']),
        ('QUBO+SA V2',     r['SRS_qubo'],  r['feas_qubo'],  r['time_build']+r['time_solve']),
    ]
    if 'SRS_qaoa' in r:
        rows.append(('QAOA (Qiskit)', r['SRS_qaoa'], r['feas_qaoa'], r['time_qaoa']))

    for name, srs, feas, t_e in rows:
        d = srs - r['SRS_hist']
        f = '✅' if feas else '❌'
        print(f'║  {name:<24} {srs:>11.6f} {d:>+11.6f} {f:>5} {t_e:>6.1f}s  ║')
    print('║' + ' '*74 + '║')

print('╠' + '═'*74 + '╣')
print(f'║  S_max={S_MAX:>12,.0f} | S_min={S_MIN:>12,.0f} | S_init={S_INIT:>12,.0f} TCM'.ljust(74) + '  ║')
print(f'║  Δu={delta_u:>10,.0f} | u_max={u_max:>10,.0f} | η={eta}'.ljust(74) + '  ║')
print('╚' + '═'*74 + '╝')

## 16. Exportar Solución

In [ ]:
r = results['Medium']
T = r['T']

sol = pd.DataFrame({
    'week': range(T),
    'R_obs': r['R_T'],
    'u_sa': r['u_sa'],
    'u_qubo': r['u_qubo'],
    'R_opt_sa': r['R_T'] + r['u_sa'],
    'R_opt_qubo': r['R_T'] + r['u_qubo'],
    'S_hist': r['S_hist'][1:],
    'S_sa': r['S_sa'][1:],
    'S_qubo': r['S_qubo'][1:],
})

sol.to_csv('falcon_solution_v2_medium.csv', index=False)
print('✅ falcon_solution_v2_medium.csv exportado')
display(sol.round(1))

try:
    from google.colab import files
    files.download('falcon_solution_v2_medium.csv')
except: pass